# Incident Response Runbook: Lateral Movement - Exploitation of Remote Services

**Tactic:** Lateral Movement
**Technique:** Exploitation of Remote Services (T1210)
**Severity:** CRITICAL

## Overview

This runbook provides step-by-step incident response procedures for Exploitation of Remote Services activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add the project root to the Python path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()

# Query Splunk for remote service exploitation indicators
print(f"\n[QUERY] Searching Splunk for remote service exploitation indicators...")
splunk_query = '''
index=* (sourcetype=WinEventLog:Security (EventCode=4625 OR EventCode=4648 OR EventCode=4672) "remote" OR "RDP" OR "SMB" OR "WMI")
OR (sourcetype=linux_secure "sshd" "Failed password" OR "Accepted password" OR "session opened")
OR (sourcetype=network "exploit" OR "vulnerability" OR "CVE-*" OR "ETERNALBLUE")
OR (sourcetype=WinEventLog:Security EventCode=5145 ShareName!="*IPC$")
| eval target_service=coalesce(ServiceName, ShareName, ProcessName)
| eval source_ip=coalesce(SourceAddress, ClientAddress, src_ip)
| stats count by host, source_ip, target_service, user, _time
| where count > 5
'''

try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-24h")
    print(f"   Found {len(splunk_results)} potential remote service exploitation events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

# Extract affected systems and exploitation attempts
affected_systems = []
exploitation_attempts = []
splunk_indicators = []

for event in splunk_results:
    system_info = {
        'hostname': event.get('host', 'unknown'),
        'source_ip': event.get('source_ip', 'unknown'),
        'target_service': event.get('target_service', ''),
        'user': event.get('user', 'unknown'),
        'attempt_count': event.get('count', 0),
        'last_seen': event.get('_time', detection_time)
    }
    affected_systems.append(system_info)

    # Extract indicators
    if event.get('source_ip') and event['source_ip'] != 'unknown':
        splunk_indicators.append({
            'type': 'ip',
            'value': event.get('source_ip'),
            'context': f"Exploitation attempts from {event.get('source_ip')} targeting {event.get('host', 'unknown')}"
        })

    if event.get('target_service'):
        splunk_indicators.append({
            'type': 'service',
            'value': event.get('target_service'),
            'context': f"Vulnerable service {event.get('target_service')} on {event.get('host', 'unknown')}"
        })

# Query CrowdStrike for lateral movement detections
print(f"\n[QUERY] Checking CrowdStrike for lateral movement detections...")
try:
    cs_detections = crowdstrike.get_detections(
        filter="technique:'Exploitation of Remote Services'",
        start_time="-24h"
    )
    cs_affected_hosts = []
    for detection in cs_detections:
        host_info = {
            'device_id': detection.get('device_id', ''),
            'hostname': detection.get('hostname', ''),
            'detection_id': detection.get('detection_id', ''),
            'technique': detection.get('technique', ''),
            'severity': detection.get('severity', 0),
            'exploit_details': detection.get('exploit_details', {})
        }
        cs_affected_hosts.append(host_info)

        # Merge with Splunk data
        existing_system = next((s for s in affected_systems if s['hostname'] == host_info['hostname']), None)
        if existing_system:
            existing_system.update(host_info)
        else:
            affected_systems.append(host_info)

    print(f"   Found {len(cs_affected_hosts)} CrowdStrike lateral movement detections")
except Exception as e:
    print(f"   CrowdStrike query failed: {e}")
    cs_affected_hosts = []

# Enrich with threat intelligence from MISP
print(f"\n[ENRICHMENT] Checking MISP for related threat intelligence...")
misp_results = []
try:
    for indicator in splunk_indicators:
        misp_search = misp.search_iocs(indicator['value'])
        if misp_search:
            misp_results.extend(misp_search)
            print(f"   Found threat intelligence for {indicator['value']}")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    incident_data = {
        'title': f'Remote Service Exploitation Incident - {len(affected_systems)} systems',
        'description': f'Detected remote service exploitation attempts on {len(affected_systems)} systems',
        'severity': 'CRITICAL',
        'tactic': 'Lateral Movement',
        'technique': 'Exploitation of Remote Services (T1210)',
        'affected_systems': affected_systems,
        'indicators': splunk_indicators,
        'threat_intel': misp_results,
        'detection_time': detection_time
    }
    incident_id = iris.create_case(incident_data)
    print(f"   Created IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"TEMP-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n Detection complete:")
print(f"  - {len(affected_systems)} affected systems identified")
print(f"  - {len(splunk_indicators)} exploitation indicators found")
print(f"  - {len(misp_results)} threat intelligence matches")
print(f"  - IRIS case {incident_id} created")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
isolated_hosts = []
blocked_ips = []
disabled_services = []

# Isolate affected systems
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            isolation_result = crowdstrike.isolate_host(system['device_id'])
            if isolation_result.get('status') == 'isolated':
                isolated_hosts.append(system['hostname'])
                print(f"   Isolated host: {system['hostname']}")
    except Exception as e:
        print(f"   Host isolation failed for {system.get('hostname', 'unknown')}: {e}")

# Block malicious source IPs
print(f"\n[ACTION] Blocking malicious source IPs...")
unique_ips = set()
for system in affected_systems:
    if system.get('source_ip') and system['source_ip'] != 'unknown':
        unique_ips.add(system['source_ip'])

for ip in unique_ips:
    try:
        block_result = shuffle.block_ip_address(ip)
        if block_result.get('status') == 'blocked':
            blocked_ips.append(ip)
            print(f"   Blocked IP: {ip}")
    except Exception as e:
        print(f"   IP blocking failed for {ip}: {e}")

# Disable vulnerable remote services
print(f"\n[ACTION] Disabling vulnerable remote services...")
unique_services = set()
for system in affected_systems:
    if system.get('target_service'):
        unique_services.add(system['target_service'])

for service in unique_services:
    try:
        disable_result = shuffle.disable_remote_service(service)
        if disable_result.get('status') == 'disabled':
            disabled_services.append(service)
            print(f"   Disabled vulnerable service: {service}")
    except Exception as e:
        print(f"   Service disable failed for {service}: {e}")

# Enable enhanced monitoring
print(f"\n[ACTION] Enabling enhanced monitoring...")
monitoring_rules = []
try:
    # Enable CrowdStrike lateral movement monitoring
    cs_monitoring = crowdstrike.enable_enhanced_monitoring('lateral_movement')
    if cs_monitoring.get('status') == 'enabled':
        monitoring_rules.append('CrowdStrike lateral movement monitoring')
        print(f"   Enabled CrowdStrike lateral movement monitoring")

    # Enable Splunk exploit detection rules
    splunk_monitoring = splunk.enable_correlation_rule('remote_exploit_detection')
    if splunk_monitoring.get('status') == 'enabled':
        monitoring_rules.append('Splunk exploit detection correlation')
        print(f"   Enabled Splunk exploit detection correlation rules")
except Exception as e:
    print(f"   Enhanced monitoring setup failed: {e}")

# Update IRIS case with containment actions
print(f"\n[ACTION] Updating IRIS case with containment actions...")
try:
    containment_summary = {
        'isolated_hosts': len(isolated_hosts),
        'blocked_ips': len(blocked_ips),
        'disabled_services': len(disabled_services),
        'monitoring_enabled': len(monitoring_rules),
        'containment_time': containment_time
    }
    iris.update_case(incident_id, 'containment_complete', containment_summary)
    print(f"   Updated IRIS case {incident_id} with containment results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_hosts)} hosts isolated")
print(f"  - {len(blocked_ips)} malicious IPs blocked")
print(f"  - {len(disabled_services)} vulnerable services disabled")
print(f"  - {len(monitoring_rules)} enhanced monitoring rules enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

removed_exploits = []
patched_services = []
cleaned_connections = []

# Remove exploit artifacts
print(f"\n[ACTION] Removing exploit artifacts...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            exploit_removal = crowdstrike.remove_exploit_artifacts(system['device_id'])
            if exploit_removal.get('status') == 'removed':
                removed_exploits.extend(exploit_removal.get('removed_files', []))
                print(f"   Removed {len(exploit_removal.get('removed_files', []))} exploit artifacts from {system['hostname']}")
    except Exception as e:
        print(f"   Exploit artifact removal failed for {system.get('hostname', 'unknown')}: {e}")

# Patch vulnerable services
print(f"\n[ACTION] Patching vulnerable services...")
for service in disabled_services:
    try:
        patch_result = shuffle.patch_vulnerable_service(service)
        if patch_result.get('status') == 'patched':
            patched_services.append(service)
            print(f"   Patched vulnerable service: {service}")
    except Exception as e:
        print(f"   Service patching failed for {service}: {e}")

# Clean malicious connections
print(f"\n[ACTION] Cleaning malicious connections...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            connection_cleaning = crowdstrike.clean_malicious_connections(system['device_id'])
            if connection_cleaning.get('status') == 'cleaned':
                cleaned_connections.append(system['hostname'])
                print(f"   Cleaned malicious connections on {system['hostname']}")
    except Exception as e:
        print(f"   Connection cleaning failed for {system.get('hostname', 'unknown')}: {e}")

# Remove backdoors and persistence
print(f"\n[ACTION] Removing backdoors and persistence...")
removed_backdoors = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            backdoor_removal = crowdstrike.remove_backdoors(system['device_id'])
            if backdoor_removal.get('status') == 'removed':
                removed_backdoors.extend(backdoor_removal.get('removed_backdoors', []))
                print(f"   Removed {len(backdoor_removal.get('removed_backdoors', []))} backdoors from {system['hostname']}")
    except Exception as e:
        print(f"   Backdoor removal failed for {system.get('hostname', 'unknown')}: {e}")

# Verify threat removal
print(f"\n[ACTION] Verifying threat removal...")
verification_results = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            verify_result = crowdstrike.verify_exploit_removal(system['device_id'])
            verification_results.append({
                'system': system['hostname'],
                'status': 'clean' if verify_result.get('exploits_detected', True) == False else 'threats_remaining',
                'remaining_indicators': verify_result.get('remaining_indicators', 0)
            })
            status = "" if verify_result.get('exploits_detected', True) == False else ""
            print(f"  {status} Verification for {system['hostname']}: {verify_result.get('remaining_indicators', 0)} exploit indicators remaining")
    except Exception as e:
        print(f"   Verification failed for {system.get('hostname', 'unknown')}: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(removed_exploits)} exploit artifacts removed")
print(f"  - {len(patched_services)} services patched")
print(f"  - {len(cleaned_connections)} systems had connections cleaned")
print(f"  - {len(removed_backdoors)} backdoors removed")
print(f"  - {len([v for v in verification_results if v['status'] == 'clean'])} systems verified clean")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

restored_systems = []
reenabled_services = []
validated_functions = []

# Restore systems from clean backups
print(f"\n[ACTION] Restoring systems from clean backups...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            restore_result = crowdstrike.restore_from_backup(system['device_id'])
            if restore_result.get('status') == 'restored':
                restored_systems.append(system['hostname'])
                print(f"   Restored {system['hostname']} from clean backup")
    except Exception as e:
        print(f"   System restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Re-enable patched services
print(f"\n[ACTION] Re-enabling patched services...")
for service in patched_services:
    try:
        enable_result = shuffle.enable_remote_service(service)
        if enable_result.get('status') == 'enabled':
            reenabled_services.append(service)
            print(f"   Re-enabled patched service: {service}")
    except Exception as e:
        print(f"   Service re-enabling failed for {service}: {e}")

# Validate system functionality
print(f"\n[ACTION] Validating system functionality...")
functional_tests = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            test_result = crowdstrike.perform_functional_testing(system['device_id'])
            functional_tests.append({
                'system': system['hostname'],
                'tests_passed': test_result.get('tests_passed', 0),
                'total_tests': test_result.get('total_tests', 0)
            })
            pass_rate = test_result.get('tests_passed', 0) / max(test_result.get('total_tests', 1), 1) * 100
            status = "" if pass_rate >= 95 else ""
            print(f"  {status} Functional tests for {system['hostname']}: {test_result.get('tests_passed', 0)}/{test_result.get('total_tests', 0)} passed ({pass_rate:.1f}%)")
    except Exception as e:
        print(f"   Functional testing failed for {system.get('hostname', 'unknown')}: {e}")

# Restore network connectivity
print(f"\n[ACTION] Restoring network connectivity...")
restored_connectivity = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            connectivity_result = crowdstrike.restore_network_connectivity(system['device_id'])
            if connectivity_result.get('status') == 'restored':
                restored_connectivity.append(system['hostname'])
                print(f"   Restored network connectivity for {system['hostname']}")
    except Exception as e:
        print(f"   Network connectivity restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Monitor for residual issues
print(f"\n[ACTION] Establishing monitoring for residual issues...")
monitoring_alerts = []
try:
    # Set up continuous monitoring
    monitoring_setup = splunk.setup_continuous_monitoring('remote_exploit_detection')
    if monitoring_setup.get('status') == 'enabled':
        monitoring_alerts.append('Splunk continuous monitoring')
        print(f"   Established continuous Splunk monitoring")

    cs_monitoring = crowdstrike.setup_residual_monitoring('lateral_movement')
    if cs_monitoring.get('status') == 'enabled':
        monitoring_alerts.append('CrowdStrike residual monitoring')
        print(f"   Established CrowdStrike residual monitoring")
except Exception as e:
    print(f"   Residual monitoring setup failed: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored from backup")
print(f"  - {len(reenabled_services)} services re-enabled")
print(f"  - {len(restored_connectivity)} systems had connectivity restored")
print(f"  - {len([f for f in functional_tests if f['tests_passed'] == f['total_tests']])} systems passed all functional tests")
print(f"  - {len(monitoring_alerts)} residual monitoring systems established")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident
print("\n" + "=" * 60)
print("STEP 5: Post-Incident")
print("=" * 60)

# Update IRIS case with eradication results
print(f"\n[ACTION] Updating IRIS case with eradication results...")
try:
    eradication_summary = {
        'removed_exploits': len(removed_exploits),
        'patched_services': len(patched_services),
        'cleaned_connections': len(cleaned_connections),
        'removed_backdoors': len(removed_backdoors),
        'verification_results': verification_results
    }
    iris.update_case(incident_id, 'eradication_complete', eradication_summary)
    print(f"   Updated IRIS case {incident_id} with eradication results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

# Update IRIS case with recovery results
print(f"\n[ACTION] Updating IRIS case with recovery results...")
try:
    recovery_summary = {
        'restored_systems': len(restored_systems),
        'reenabled_services': len(reenabled_services),
        'functional_tests': functional_tests,
        'restored_connectivity': len(restored_connectivity),
        'monitoring_alerts': len(monitoring_alerts)
    }
    iris.update_case(incident_id, 'recovery_complete', recovery_summary)
    print(f"   Updated IRIS case {incident_id} with recovery results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

# Generate incident report
print(f"\n[ACTION] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'technique': 'Lateral Movement: Exploitation of Remote Services',
        'detection_time': detection_time,
        'affected_systems': len(affected_systems),
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': datetime.now().isoformat(),
            'recovery': datetime.now().isoformat()
        },
        'actions_taken': {
            'detection': f"Found {len(affected_systems)} systems with remote service exploitation",
            'containment': f"Isolated {len(isolated_hosts)} hosts, blocked {len(blocked_ips)} IPs",
            'eradication': f"Removed {len(removed_exploits)} exploits, patched {len(patched_services)} services",
            'recovery': f"Restored {len(restored_systems)} systems, re-enabled {len(reenabled_services)} services"
        },
        'indicators': splunk_indicators,
        'threat_intel': misp_results,
        'lessons_learned': [
            "Implement stricter remote service access controls",
            "Regular vulnerability scanning and patching of remote services",
            "Enhanced monitoring of lateral movement attempts"
        ]
    }
    iris.generate_report(incident_id, incident_report)
    print(f"   Generated comprehensive incident report for case {incident_id}")
except Exception as e:
    print(f"   Report generation failed: {e}")

# Share IOCs with MISP
print(f"\n[ACTION] Sharing IOCs with MISP...")
try:
    misp_iocs = []
    for indicator in splunk_indicators:
        if indicator.get('type') in ['ip', 'service']:
            misp_iocs.append({
                'type': indicator['type'],
                'value': indicator['value'],
                'tags': ['remote-service-exploit', 'incident-' + str(incident_id)]
            })

    if misp_iocs:
        misp.share_iocs(misp_iocs, f"Remote Service Exploitation Incident {incident_id}")
        print(f"   Shared {len(misp_iocs)} IOCs with MISP community")
    else:
        print(f"   No new IOCs to share with MISP")
except Exception as e:
    print(f"   MISP IOC sharing failed: {e}")

# Close IRIS case
print(f"\n[ACTION] Closing IRIS case...")
try:
    iris.close_case(incident_id, 'resolved')
    print(f"   Closed IRIS case {incident_id} as resolved")
except Exception as e:
    print(f"   IRIS case closure failed: {e}")

# Final status summary
print(f"\n Post-incident activities complete:")
print(f"  - IRIS case {incident_id} updated and closed")
print(f"  - Incident report generated")
print(f"  - {len(misp_iocs) if 'misp_iocs' in locals() else 0} IOCs shared with threat intelligence community")
print(f"  - All 5 IR phases completed successfully")

# Calculate incident metrics
incident_duration = (datetime.now() - datetime.fromisoformat(detection_time)).total_seconds() / 3600
print(f"\n📊 Incident Metrics:")
print(f"  - Duration: {incident_duration:.2f} hours")
print(f"  - Systems affected: {len(affected_systems)}")
print(f"  - Containment time: {'< 1 hour' if incident_duration < 1 else f'{incident_duration:.1f} hours'}")
print(f"  - Eradication success: {len([v for v in verification_results if v['status'] == 'clean'])}/{len(affected_systems)} systems clean")
print(f"  - Recovery success: {len([f for f in functional_tests if f['tests_passed'] == f['total_tests']])}/{len(affected_systems)} systems fully functional")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
